<a href="https://colab.research.google.com/github/lakshya701/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakshya701/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
## Task Type: Scoring (built on binary classification)

My lane, Refresh / Content Opportunity Scoring, is fundamentally a **scoring/ranking**
problem, built on top of a **binary classification** model. The classifier predicts
a probability that a page is declining; that probability becomes the score used to
rank pages, so a reviewer can act on the top of the list first.

It isn't clustering I'm not looking for unlabeled groups of similar pages. It isn't
pure ranking-without-a-label either I do have a defined outcome (declining vs. not)
that a classifier learns to predict, and I convert its probability output into a ranked
queue. That combination  classify, then rank by the classifier's confidence  is
exactly what "scoring" means in this lane guide.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
## Target / Proxy

**Current proxy (starter data):** `is_declining_label = trend_direction == "down"`.
This is a *current-window bucket*, not an observed future outcome — it's calculated
from the same window I'd be using as features, which makes it a weak proxy (as the
lane guide itself warns).

**Where the label comes from:** It's a defined rule applied to an existing column
(`trend_direction`), not a directly observed future event.

**Stronger target I'm working toward:** A future-window label — using the prior 90
days of features to predict whether impressions decline by more than a set threshold
(e.g. 20%) over the *next* 30 days. This avoids the label being calculated from the
same window as the features, which is the leakage risk the lane guide flags.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*
## Success Metric: Precision@50

I'm choosing **Precision@50** as my defendable metric, not accuracy or AUC.

**Why:** A reviewer only has time to check a limited number of flagged pages — the
lane guide frames this explicitly as "a reviewer checks 20 pages" / "the team can
act on 50 candidates." Precision@50 answers exactly the question that matters here:
*of the top 50 pages my system flags, how many are actually declining?* That's the
real-world cost — wasted reviewer time on false positives — not overall classification
accuracy across all 30,000 pages, most of which nobody will ever look at.

**What "good" means:** My own Notebook 02 run already beats a hand-rule baseline
(hand rule Precision@50 = 0.600, depth-3 tree = 0.720). The starter benchmark shows
random forest reaching 0.740 against a rule baseline of 0.240. Good, for this lane,
means consistently beating the transparent hand-rule baseline on held-out clients —
not hitting some absolute score.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
## Unit of Analysis

One row = one content page, evaluated at a single point in time using its trailing
performance window. Below is the actual dataframe slice, with the columns that would
become features, and a sketch of what the target column looks like.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lakshya701/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Sketch the target column
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# One row = one content page — show the unit of analysis with the columns that matter
cols = ["content_id", "impressions_90d", "days_since_last_update",
        "avg_position", "ctr", "trend_direction", "is_declining_label"]
df[cols].head(10)

,content_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,19140,14,44.0,0.13,down,1
5,content_d4084a4bc775,3970,20,8.5,0.03,down,1
6,content_9a34b442b552,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,1724,22,21.2,0.06,stable,0
8,content_5e6c160719bc,32574,20,46.0,0.09,down,1
9,content_c27558df2b0c,1240,104,4.9,0.16,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
## Why ML Beats a Fixed Rule Here

A hand rule like `stale AND visible` can only combine a few conditions with a single
logic gate — it can't weigh *how much* staleness matters relative to *how much*
visibility, or capture that the relationship changes depending on position or content
age. I already have direct proof of this from Notebook 02: my hand-written rule
(`stale x visible`) hit Precision@50 of 0.600, while a depth-3 decision tree — which
can combine multiple thresholds across several features and weigh them — reached
0.720 on the exact same data.

The pattern is "too messy for an if-statement" because staleness, visibility, position,
and CTR interact — a page can be stale but still fine if it's visible and well-ranked,
or fresh but declining if visibility collapsed. A single rule can't express that kind
of conditional interaction across four+ variables at once; a tree or forest learns
exactly which combinations matter, and by how much, directly from the data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.